# Phase 1 — Centralized TB Classifier (Colab)

Trains a DenseNet121 TB classifier on **Shenzhen + Montgomery** and locks in the metric harness.

**Before running:** set Runtime → Change runtime type → **GPU**.

**Data layout** (in your Google Drive, e.g. `MyDrive/tb-fl/data/raw/`):
```
shenzhen/    (ChinaSet_AllFiles: CXR_png/ + ClinicalReadings/)
montgomery/  (MontgomerySet: CXR_png/ + ClinicalReadings/)
```

## 1. Mount Drive + clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone your repo (replace with your remote if private — use a token or GitHub auth).
!git clone https://github.com/karansharma1732/project-dheeru-ki-phd.git
%cd project-dheeru-ki-phd/code

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. (Optional) edit config
Confirm the dataset paths in `config.yaml` point to your Drive folders. Adjust epochs / batch size for the Colab GPU if needed.

In [ ]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['paths']

## 4. Parse demographics → demographics.csv

In [ ]:
!python parse_demographics.py --config config.yaml

In [ ]:
# Quick look at class + subgroup balance.
import pandas as pd
df = pd.read_csv(cfg['paths']['demographics_csv'])
print(df.groupby(['dataset','label']).size())
print(df['sex'].value_counts())
print(df['age_band'].value_counts())

## 5. Train + evaluate

In [ ]:
!python train.py --config config.yaml

## 6. Grad-CAM sanity check

In [ ]:
out_dir = cfg['paths']['out_dir']
!python gradcam.py --config config.yaml --checkpoint "{out_dir}/best.pt" --n 8

In [ ]:
# Display a couple of overlays.
import glob
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob(f"{out_dir}/gradcam/*.png"))[:4]:
    display(IPImage(p))